## Fraud Detection

## STEP 1 — Import Libraries

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## STEP 2 — Load & Understand the Data

In [ ]:
df = pd.read_csv('fraud_detection_dataset.csv')

print(df.head())            # first 5 rows

In [ ]:
print(df.shape)             # (1000, 7)

In [ ]:
print(df.dtypes)            # column types

In [ ]:
print(df.describe())        # statistics

In [ ]:
print(df.isnull().sum())    # missing values


In [ ]:
print(df['Fraud'].value_counts())  # how many fraud vs not fraud

## STEP 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Fraud vs Not Fraud count
plt.figure(figsize=(5,4))
df['Fraud'].value_counts().plot(kind='bar', color=['steelblue','crimson'], edgecolor='white')
plt.title('Fraud vs Not Fraud')
plt.xlabel('0 = Not Fraud | 1 = Fraud')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Amount distribution by Fraud
plt.figure(figsize=(7,4))
df[df['Fraud']==0]['Amount'].hist(bins=30, alpha=0.6, color='steelblue', label='Not Fraud')
df[df['Fraud']==1]['Amount'].hist(bins=30, alpha=0.6, color='crimson', label='Fraud')
plt.title('Amount Distribution — Fraud vs Not Fraud')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# Fraud count by Location
plt.figure(figsize=(7,4))
df.groupby('Location')['Fraud'].sum().sort_values(ascending=False).plot(
    kind='bar', color='coral', edgecolor='white')
plt.title('Fraud Count by Location')
plt.ylabel('Number of Frauds')
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Fraud count by DeviceType
plt.figure(figsize=(5,4))
df.groupby('DeviceType')['Fraud'].sum().plot(
    kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Fraud Count by Device Type')
plt.ylabel('Number of Frauds')
plt.xticks(rotation=0)
plt.show()

## STEP 4 — Data Preprocessing

In [ ]:
# Drop TransactionID — just a serial number, useless for prediction
df = df.drop(columns=['TransactionID'])

# One-Hot Encode all 3 categorical columns
df = pd.get_dummies(df, columns=['Location', 'Merchant', 'DeviceType'], drop_first=True)



In [ ]:
print(df.head())

In [ ]:
print(df.shape)

In [ ]:
print(df.columns.tolist())

## STEP 5 — Split Features & Target, Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Fraud'])   # all columns except Fraud
y = df['Fraud']                  # only the Fraud column

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows :", X_train.shape[0])
print("Testing rows  :", X_test.shape[0])

## STEP 6 — Build & Train the Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Note: Using Classifiers now (not Regressors like house prices!)
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully")

## STEP 7 — Evaluate the Models

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for name, model in models.items():
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"\n {name}")
    print(f"   Accuracy : {acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

## STEP 8 — Confusion Matrix (Visual)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

best_model = models['Random Forest']
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Fraud','Fraud'],
            yticklabels=['Not Fraud','Fraud'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## STEP 9 — Feature Importance

In [ ]:
import pandas as pd

best_model = models['Random Forest']

feat_imp = pd.Series(best_model.feature_importances_,
                     index=X.columns).sort_values(ascending=True)

feat_imp.plot(kind='barh', figsize=(8,6), color='steelblue', edgecolor='white')
plt.title('Feature Importance — What causes Fraud?')
plt.xlabel('Importance Score')
plt.show()

## STEP 10 — Predict on a New Transaction

In [ ]:
# Example: New transaction to check if it's fraud
new_transaction = pd.DataFrame([{
    'Amount': 95000,
    'Time'  : 2,
}])

# Add all the encoded columns with 0s (adjust based on your actual columns)
for col in X.columns:
    if col not in new_transaction.columns:
        new_transaction[col] = 0

new_transaction = new_transaction[X.columns]

prediction = best_model.predict(new_transaction)[0]
print(" Transaction Result:", " FRAUD!" if prediction == 1 else " Not Fraud")